# Cascadia Scoring — YOLO Model Training

Trains a YOLO detector on the synthetic dataset generated by `tools/dataset_genetor.py`.

Prerequisites:
1. Generate the dataset first, e.g. `python tools/dataset_genetor.py -train 100 -val 20`.
2. The dataset lives in `datasets/for_model/{train,val}/{images,labels}`.

Run the cells from top to bottom.

In [ ]:
!pip install -q ultralytics

## 1. Select architecture

Edit `MODEL_NAME` in the next cell to pick a different YOLO architecture.

| Model | Sizes |
|---|---|
| YOLO11 | `yolo11n` `yolo11s` `yolo11m` `yolo11l` `yolo11x` |
| YOLOv8 | `yolov8n` `yolov8s` `yolov8m` `yolov8l` `yolov8x` |

`n` is the fastest and smallest; `x` is the biggest and most accurate. Pretrained COCO weights are downloaded automatically on first use.

In [ ]:
# Select a YOLO architecture:
#   YOLO11 : yolo11n, yolo11s, yolo11m, yolo11l, yolo11x
#   YOLOv8 : yolov8n, yolov8s, yolov8m, yolov8l, yolov8x
MODEL_NAME = "yolo11n"

## 2. Hyperparameters

Tweak these in the next cell. With a small dataset, use a small `BATCH` and consider fewer `EPOCHS`.

In [ ]:
import torch

EPOCHS = 100
IMGSZ = 640
BATCH = 8
WORKERS = 2
DEVICE = 0 if torch.cuda.is_available() else "cpu"
PROJECT = "runs"
NAME = "cascadia"

print(f"Training on device: {DEVICE}")

## 3. Train

Downloads the pretrained checkpoint on first run, then fine-tunes on the generated dataset. Results land in `runs/cascadia/`.

In [ ]:
from ultralytics import YOLO

data_yaml = "/config.yml"

model = YOLO(f"{MODEL_NAME}.pt")

model.train(
    data=data_yaml,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    workers=WORKERS,
    device=DEVICE,
    project=PROJECT,
    name=NAME,
)

## 4. Evaluate

Runs the trained model on the validation split and prints mAP / precision / recall.

In [ ]:
metrics = model.val()

print(f"mAP50:   {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

## 5. Save & test

Copies the best weights into `output/` and shows a quick prediction on a validation image.

In [ ]:
import shutil

best_path = model.trainer.best
output_dir = (NOTEBOOK_DIR / ".." / "output").resolve()
output_dir.mkdir(parents=True, exist_ok=True)
out_model = output_dir / f"cascadia_{MODEL_NAME}.pt"
shutil.copy(best_path, out_model)
print(f"Best weights saved to {out_model}")

In [ ]:
import matplotlib.pyplot as plt

results = model.predict(source=val_img_dir, imgsz=IMGSZ, conf=0.25, verbose=False)

for r in results[:1]:
    im = r.plot()
    plt.figure(figsize=(10, 10))
    plt.imshow(im[..., ::-1])
    plt.axis("off")
    plt.show()